### CONTENT BASED FILTERING

In [7]:
import pandas as pd
import numpy as np
import warnings

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [8]:
df = pd.read_csv('../data/processed/movie_features.csv')
df.columns

Index(['movieId', 'title', 'genres', 'rating', 'rating_count'], dtype='str')

In [9]:
df["genres"] = df["genres"].str.replace("Sci-Fi", "SciFi", regex=False)

df["genres"] = df["genres"].str.replace("Film-Noir", "FilmNoir", regex=False)

df["genres"] = df["genres"].str.replace("|", " ", regex=False)

df["genres"] = df["genres"].fillna("")

df["genres"] = df["genres"].replace("(no genres listed)", "")

df["genres"] = df["genres"].str.lower()

print(df.genres)

0       adventure animation children comedy fantasy
1                        adventure children fantasy
2                                    comedy romance
3                              comedy drama romance
4                                            comedy
                           ...                     
9995                                  action comedy
9996                  action adventure comedy crime
9997              adventure children comedy fantasy
9998                          action crime thriller
9999                                          drama
Name: genres, Length: 10000, dtype: str


In [10]:
# tf-idf
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,1),
    min_df=1
)

# fit and transform
tfidf_matrix = tfidf.fit_transform(df["genres"])

# get feature names
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Toplam Feature Sayısı:", len(tfidf.get_feature_names_out()))

TF-IDF Matrix Shape: (10000, 19)
Toplam Feature Sayısı: 19


In [12]:
# Cosine Similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Dimesion checking
print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

Cosine Similarity Matrix Shape: (10000, 10000)


In [19]:
# Recommendation Function
def recommend(movie_name, top_n=10):
    """
    Recommends movies similar to the given movie.

    Parameters:
        movie_name (str): Movie name entered by the user
        top_n (int): Number of recommendations to return

    Returns:
        DataFrame
    """

    # Searching for the movie
    matches = df[df["title"].str.contains(movie_name, case=False, na=False)]

    # If the movie isn't found
    if matches.empty:
        print("Movie not found.")
        return

    # Take the first matched movie
    movie_index = matches.index[0]
    selected_title = df.iloc[movie_index]["title"]

    print(f"Selected Movie: {selected_title}\n")

    # Similarity scores
    similarity_scores = list(enumerate(cosine_sim[movie_index]))

    # Sort from highest to lowest
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove itself (first result)
    similarity_scores = similarity_scores[1:top_n+1]

    # Recommended movie indexes
    movie_indices = [i[0] for i in similarity_scores]

    # Result dataframe
    recommendations = df.iloc[movie_indices][["title", "genres"]].copy()

    # Add similarity score
    recommendations["similarity_score"] = [
        round(i[1], 3) for i in similarity_scores
    ]

    return recommendations.reset_index(drop=True)

In [28]:
recommend("Interstellar",10)
# Yildizlararasi die bi film var o da cok guzel ama onermedi  :(

Selected Movie: Interstellar (2014)



,title,genres,similarity_score
0,Cloud Atlas (2012),drama scifi imax,0.963
1,Transcendence (2014),drama scifi imax,0.963
2,Contagion (2011),scifi thriller imax,0.919
3,Gravity (2013),action scifi imax,0.911
4,The Amazing Spider-Man 2 (2014),action scifi imax,0.911
5,Edge of Tomorrow (2014),action scifi imax,0.911
6,"Day the Earth Stood Still, The (2008)",drama scifi thriller imax,0.890
7,Real Steel (2011),action drama scifi imax,0.883
8,Elysium (2013),action drama scifi imax,0.883
9,Men in Black III (M.III.B.) (M.I.B.³) (2012),action comedy scifi imax,0.873
